In [230]:
!pip install catboost prophet

In [231]:
import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

import catboost as cb
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report
from pathlib import Path

In [232]:
# ## 1. Загрузка и подготовка исходных данных
#base_path = Path('..')

#inc_path =  (base_path / 'data/raw/incidents_2000.csv').resolve()
#thr_path = (base_path / 'data/raw/thrlist.csv').resolve()
inc_path = '/incidents_2000.csv'
thr_path = '/thrlist.csv'
inc = pd.read_csv(inc_path)
thr = pd.read_csv(thr_path, header=1)

df = inc.merge(
     thr,
     how="left",
     left_on="Код реализованной угрозы",
     right_on="Идентификатор УБИ"
)

In [233]:
import pandas as pd
from io import StringIO

data = """код_угрозы;кластер
1;3
2;5
3;6
4;1
5;1
6;4
7;4
8;6
9;1
10;4
11;5
12;2
13;1
14;3
15;6
16;5
17;5
18;1
19;5
20;2
21;2
22;3
23;1
24;1
25;6
26;6
27;1
28;6
29;3
30;2
31;2
32;1
33;2
34;6
35;1
36;2
37;6
38;3
39;1
40;2
41;5
42;5
43;3
44;4
45;4
46;5
47;3
48;4
49;6
50;6
51;3
52;2
53;1
54;2
55;5
56;2
57;6
58;3
59;3
60;3
61;3
62;5
63;2
64;2
65;2
66;2
67;2
68;4
69;5
70;2
71;6
72;1
73;5
74;3
75;5
76;4
77;4
78;4
79;4
80;4
81;6
82;3
83;5
84;6
85;5
86;3
87;1
88;6
89;6
90;3
91;3
92;1
93;6
94;4
95;4
96;2
97;6
98;5
99;5
100;2
101;6
102;4
103;5
104;5
105;3
106;3
107;1
108;4
109;2
110;3
111;5
112;4
113;3
114;4
115;5
116;5
117;4
118;4
119;4
120;4
121;4
122;4
123;1
124;6
125;5
126;5
127;2
128;5
129;1
130;5
131;5
132;5
133;5
134;2
135;6
136;6
137;2
138;2
139;1
140;3
141;2
142;2
143;3
144;1
145;2
146;4
147;4
148;3
149;4
150;1
151;5
152;3
153;3
154;1
155;3
156;2
157;1
158;3
159;2
160;1
161;3
162;4
163;4
164;1
165;2
166;2
167;3
168;2
169;4
170;3
171;3
172;3
173;3
174;5
175;5
176;3
177;2
178;4
179;6
180;1
181;5
182;1
183;5
184;6
185;2
186;5
187;4
188;4
189;4
190;5
191;4
192;2
193;6
194;1
195;1
196;2
197;6
198;3
199;5
200;6
201;6
202;4
203;1
204;4
205;3
206;1
207;1
208;3
209;1
210;4
211;2
212;4
213;4
214;2
215;6
216;4
217;4
218;1
219;1
220;1
221;1
222;1
223;4
224;4
225;4
226;6
227;6"""

# Создаем датафрейм
df_clasters = pd.read_csv(StringIO(data), sep=';')
# Присоединяем кластер угрозы по коду угрозы
df = df.merge(
    df_clasters,
    how='left',
    left_on='Код реализованной угрозы',
    right_on='код_угрозы'
)

df = df.dropna(subset=['код_угрозы']).copy()
df['threat_cluster'] = df['кластер'].astype(int)

In [234]:
df['threat_cluster']

,threat_cluster
0,5
1,3
2,4
3,4
4,1
...,...
1995,2
1996,1
1997,1
1998,2


In [235]:
date_cols = [
    "Региональное время",
    "Дата инцидента",
    "Дата включения угрозы в БнД УБИ",
    "Дата последнего изменения данных"
    ]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

In [236]:
# структура кластеров
infrastructure_clusters = {
    1: [ #'Digital-Native (Высокотехнологичные)'
        'информационные технологии',
        'реклама',
        'маркетинг',
        'телекоммуникации',
        'системная интеграция'
    ],

    2: [ # 'Industrial IoT (Промышленные)'
        'машиностроение',
        'металлургия',
        'химия',
        'добыча',
        'энергетика',
        'логистика'
    ],

    3: [ # 'Data-Sensitive (Чувствительные данные)'
        'медицина',
        'фармацевтика',
        'образование',
        'НКО',
        'безопасность'
    ],

    4: [ # 'Service-Oriented (Сервисные)'
        'розничная торговля',
        'отели',
        'ресторанный бизнес',
        'консалтинг',
        'автобизнес'
    ]
}

industry_to_cluster = {}
for cluster, industries in infrastructure_clusters.items():
    for industry in industries:
        industry_to_cluster[industry] = cluster


df['infrastructure_cluster'] = df['Тип предприятия'].map(industry_to_cluster)

In [237]:
# Нормализуем типы
df['infrastructure_cluster'] = df['infrastructure_cluster'].astype(str)
df['Тип предприятия'] = df['Тип предприятия'].astype(str)
df['Регион размещения предприятия'] = df['Регион размещения предприятия'].astype(str)

# Дата события без времени
df['date'] = df['Региональное время'].dt.floor('D')

df = df.sort_values(['infrastructure_cluster', 'Региональное время']).reset_index(drop=True)

#  Агрегация: infrastructure_cluster × threat_cluster × day

In [238]:
daily_entity = (
    df.groupby(['infrastructure_cluster', 'threat_cluster', 'date'], as_index=False)
      .agg({
          'Тип предприятия': 'last',
          'Регион размещения предприятия': 'last',
          'Количество хостов': 'last',
          'Успех': 'sum',
          'Региональное время': 'count'
      })
      .rename(columns={
          'Региональное время': 'incidents_count_day',
          'Успех': 'success_count_day'
      })
)

daily_entity['had_incident_today'] = (daily_entity['incidents_count_day'] > 0).astype(int)

print(daily_entity.head())
print(daily_entity.shape)

  infrastructure_cluster  threat_cluster       date  \
0                      1               1 2023-01-25   
1                      1               1 2023-03-16   
2                      1               1 2023-03-17   
3                      1               1 2023-03-18   
4                      1               1 2023-04-04   

             Тип предприятия    Регион размещения предприятия  \
0           телекоммуникации  Ямало-Ненецкий автономный округ   
1  информационные технологии             Чувашская Республика   
2                  маркетинг                  Томская область   
3  информационные технологии               Республика Хакасия   
4       системная интеграция                Кировская область   

   Количество хостов  success_count_day  incidents_count_day  \
0                390                  0                    1   
1               1015                  1                    1   
2               1559                  0                    1   
3               1774  

# Построение полного календаря по дням для каждого предприятия

In [239]:
min_date = daily_entity['date'].min()
max_date = daily_entity['date'].max()
all_dates = pd.date_range(min_date, max_date, freq='D')

entity_info = (
    df.sort_values('Региональное время')
      .groupby(['infrastructure_cluster', 'threat_cluster'], as_index=False)
      .agg({
          'Тип предприятия': 'last',
          'Регион размещения предприятия': 'last',
          'Количество хостов': 'last'
      })
)

calendar = (
    entity_info.assign(key=1)
    .merge(pd.DataFrame({'date': all_dates, 'key': 1}), on='key')
    .drop(columns='key')
)

print(calendar.head())
print(calendar.shape)

  infrastructure_cluster  threat_cluster Тип предприятия  \
0                      1               1       маркетинг   
1                      1               1       маркетинг   
2                      1               1       маркетинг   
3                      1               1       маркетинг   
4                      1               1       маркетинг   

  Регион размещения предприятия  Количество хостов       date  
0              Амурская область               1684 2023-01-01  
1              Амурская область               1684 2023-01-02  
2              Амурская область               1684 2023-01-03  
3              Амурская область               1684 2023-01-04  
4              Амурская область               1684 2023-01-05  
(26328, 6)


# Присоединяем дневную активность, заполняем пропуски нулями

In [240]:
dataset = calendar.merge(
    daily_entity[
        ['infrastructure_cluster', 'threat_cluster', 'date',
         'incidents_count_day', 'success_count_day', 'had_incident_today']
    ],
    on=['infrastructure_cluster', 'threat_cluster', 'date'],
    how='left'
)

dataset['incidents_count_day'] = dataset['incidents_count_day'].fillna(0).astype(int)
dataset['success_count_day'] = dataset['success_count_day'].fillna(0).astype(int)
dataset['had_incident_today'] = dataset['had_incident_today'].fillna(0).astype(int)

dataset = dataset.sort_values(
    ['infrastructure_cluster', 'threat_cluster', 'date']
).reset_index(drop=True)

print(dataset.head(10))

  infrastructure_cluster  threat_cluster Тип предприятия  \
0                      1               1       маркетинг   
1                      1               1       маркетинг   
2                      1               1       маркетинг   
3                      1               1       маркетинг   
4                      1               1       маркетинг   
5                      1               1       маркетинг   
6                      1               1       маркетинг   
7                      1               1       маркетинг   
8                      1               1       маркетинг   
9                      1               1       маркетинг   

  Регион размещения предприятия  Количество хостов       date  \
0              Амурская область               1684 2023-01-01   
1              Амурская область               1684 2023-01-02   
2              Амурская область               1684 2023-01-03   
3              Амурская область               1684 2023-01-04   
4             

# Target: будет ли инцидент в следующие 24 часа

In [241]:
dataset['target_next_24h'] = (
    dataset.groupby(['infrastructure_cluster', 'threat_cluster'])['had_incident_today']
           .shift(-1)
)

dataset = dataset.dropna(subset=['target_next_24h']).copy()
dataset['target_next_24h'] = dataset['target_next_24h'].astype(int)

print(dataset[['infrastructure_cluster', 'threat_cluster', 'date',
               'had_incident_today', 'target_next_24h']].head(10))

  infrastructure_cluster  threat_cluster       date  had_incident_today  \
0                      1               1 2023-01-01                   0   
1                      1               1 2023-01-02                   0   
2                      1               1 2023-01-03                   0   
3                      1               1 2023-01-04                   0   
4                      1               1 2023-01-05                   0   
5                      1               1 2023-01-06                   0   
6                      1               1 2023-01-07                   0   
7                      1               1 2023-01-08                   0   
8                      1               1 2023-01-09                   0   
9                      1               1 2023-01-10                   0   

   target_next_24h  
0                0  
1                0  
2                0  
3                0  
4                0  
5                0  
6                0  
7     

In [242]:
def make_future_target_next_k_days(group, source_col='had_incident_today', k=7):
    s = group[source_col]
    future = pd.concat([s.shift(-i) for i in range(1, k + 1)], axis=1)
    return future.max(axis=1)

dataset['target_next_7d'] = (
    dataset.groupby(['infrastructure_cluster', 'threat_cluster'], group_keys=False)
           .apply(make_future_target_next_k_days, source_col='had_incident_today', k=7)
)

dataset = dataset.dropna(subset=['target_next_7d']).copy()
dataset['target_next_7d'] = dataset['target_next_7d'].astype(int)

print("Доля positives target_next_24h:", dataset['target_next_24h'].mean())
print("Доля positives target_next_7d:", dataset['target_next_7d'].mean())

Доля positives target_next_24h: 0.07245053272450533
Доля positives target_next_7d: 0.40859969558599696


# Календарные признаки

In [243]:
dataset['day_of_week'] = dataset['date'].dt.dayofweek
dataset['day_of_month'] = dataset['date'].dt.day
dataset['month'] = dataset['date'].dt.month
dataset['quarter'] = dataset['date'].dt.quarter
dataset['year'] = dataset['date'].dt.year
dataset['is_weekend'] = (dataset['day_of_week'] >= 5).astype(int)

dataset['dow_sin'] = np.sin(2 * np.pi * dataset['day_of_week'] / 7)
dataset['dow_cos'] = np.cos(2 * np.pi * dataset['day_of_week'] / 7)

dataset['month_sin'] = np.sin(2 * np.pi * dataset['month'] / 12)
dataset['month_cos'] = np.cos(2 * np.pi * dataset['month'] / 12)

# Исторические признаки по предприятию

In [244]:
grp = dataset.groupby(['infrastructure_cluster', 'threat_cluster'])

dataset['lag_inc_1d'] = grp['incidents_count_day'].shift(1).fillna(0)
dataset['lag_inc_2d'] = grp['incidents_count_day'].shift(2).fillna(0)
dataset['lag_inc_3d'] = grp['incidents_count_day'].shift(3).fillna(0)

dataset['lag_success_1d'] = grp['success_count_day'].shift(1).fillna(0)
dataset['lag_success_2d'] = grp['success_count_day'].shift(2).fillna(0)


# Rolling counts

In [245]:
'''dataset['inc_3d_sum'] = (
    grp['incidents_count_day']
    .shift(1)
    .rolling(3, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

dataset['inc_7d_sum'] = (
    grp['incidents_count_day']
    .shift(1)
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

dataset['inc_30d_sum'] = (
    grp['incidents_count_day']
    .shift(1)
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

dataset['succ_7d_sum'] = (
    grp['success_count_day']
    .shift(1)
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

dataset['succ_30d_sum'] = (
    grp['success_count_day']
    .shift(1)
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

dataset['had_incident_prev_1d'] = (dataset['lag_inc_1d'] > 0).astype(int)
dataset['had_incident_prev_3d'] = (dataset['inc_3d_sum'] > 0).astype(int)
dataset['had_incident_prev_7d'] = (dataset['inc_7d_sum'] > 0).astype(int)'''

# Функция для безопасного сброса всех уровней индекса
def safe_reset(series):
    """Сбрасывает все уровни индекса, сколько бы их ни было"""
    return series.reset_index(drop=True)
'''
# Применяем для всех признаков
dataset['inc_3d_sum'] = safe_reset(grp['incidents_count_day'].shift(1).rolling(3, min_periods=1).sum())
dataset['inc_7d_sum'] = safe_reset(grp['incidents_count_day'].shift(1).rolling(7, min_periods=1).sum())
dataset['inc_30d_sum'] = safe_reset(grp['incidents_count_day'].shift(1).rolling(30, min_periods=1).sum())
dataset['succ_7d_sum'] = safe_reset(grp['success_count_day'].shift(1).rolling(7, min_periods=1).sum())
dataset['succ_30d_sum'] = safe_reset(grp['success_count_day'].shift(1).rolling(30, min_periods=1).sum())

dataset['had_incident_prev_1d'] = (dataset['lag_inc_1d'] > 0).astype(int)
dataset['had_incident_prev_3d'] = (dataset['inc_3d_sum'] > 0).astype(int)
dataset['had_incident_prev_7d'] = (dataset['inc_7d_sum'] > 0).astype(int)'''

def add_rolling(part):
    part['inc_3d_sum'] = safe_reset(
        grp['incidents_count_day'].shift(1).rolling(3,  min_periods=1).sum()
    )
    part['inc_7d_sum'] = safe_reset(
        grp['incidents_count_day'].shift(1).rolling(7,  min_periods=1).sum()
    )
    part['inc_30d_sum'] = safe_reset(
        grp['incidents_count_day'].shift(1).rolling(30, min_periods=1).sum()
    )
    part['succ_7d_sum'] = safe_reset(
        grp['success_count_day'].shift(1).rolling(7,  min_periods=1).sum()
    )
    part['succ_30d_sum'] = safe_reset(
        grp['success_count_day'].shift(1).rolling(30, min_periods=1).sum()
    )

    part['had_incident_prev_1d'] = (part['lag_inc_1d'] > 0).astype(int)
    part['had_incident_prev_3d'] = (part['inc_3d_sum'] > 0).astype(int)
    part['had_incident_prev_7d'] = (part['inc_7d_sum'] > 0).astype(int)
    return part

# Время с последнего инцидента

In [246]:
def days_since_last_incident(series):
    result = []
    last_day = None
    for i, val in enumerate(series):
        if last_day is None:
            result.append(np.nan)
        else:
            result.append(i - last_day)
        if val > 0:
            last_day = i
    return pd.Series(result, index=series.index)

dataset['days_since_last_incident'] = (
    dataset.groupby(['infrastructure_cluster', 'threat_cluster'])['had_incident_today']
           .apply(days_since_last_incident)
           .reset_index(level=[0,1], drop=True)
)
dataset['days_since_last_incident'] = dataset['days_since_last_incident'].fillna(999)

dataset['had_success_today'] = (dataset['success_count_day'] > 0).astype(int)

dataset['days_since_last_success'] = (
    dataset.groupby(['infrastructure_cluster', 'threat_cluster'])['had_success_today']
           .apply(days_since_last_incident)
           .reset_index(level=[0,1], drop=True)
)
dataset['days_since_last_success'] = dataset['days_since_last_success'].fillna(999)

# Исторические признаки по региону и типу предприятия

По региону


In [247]:
region_day = (
    dataset.groupby(['Регион размещения предприятия', 'date'], as_index=False)
           .agg({
               'incidents_count_day': 'sum',
               'success_count_day': 'sum'
           })
           .rename(columns={
               'incidents_count_day': 'region_incidents_day',
               'success_count_day': 'region_success_day'
           })
)

region_day = region_day.sort_values(['Регион размещения предприятия', 'date'])
grp_r = region_day.groupby('Регион размещения предприятия')

region_day['region_inc_7d_sum'] = (
    grp_r['region_incidents_day']
    .shift(1)
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

region_day['region_inc_30d_sum'] = (
    grp_r['region_incidents_day']
    .shift(1)
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

dataset = dataset.merge(
    region_day[['Регион размещения предприятия', 'date', 'region_inc_7d_sum', 'region_inc_30d_sum']],
    on=['Регион размещения предприятия', 'date'],
    how='left'
)

dataset['region_inc_7d_sum'] = dataset['region_inc_7d_sum'].fillna(0)
dataset['region_inc_30d_sum'] = dataset['region_inc_30d_sum'].fillna(0)

По типу предприятия

In [248]:
type_day = (
    dataset.groupby(['Тип предприятия', 'date'], as_index=False)
           .agg({
               'incidents_count_day': 'sum',
               'success_count_day': 'sum'
           })
           .rename(columns={
               'incidents_count_day': 'type_incidents_day',
               'success_count_day': 'type_success_day'
           })
)

type_day = type_day.sort_values(['Тип предприятия', 'date'])
grp_t = type_day.groupby('Тип предприятия')

type_day['type_inc_7d_sum'] = (
    grp_t['type_incidents_day']
    .shift(1)
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

type_day['type_inc_30d_sum'] = (
    grp_t['type_incidents_day']
    .shift(1)
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

dataset = dataset.merge(
    type_day[['Тип предприятия', 'date', 'type_inc_7d_sum', 'type_inc_30d_sum']],
    on=['Тип предприятия', 'date'],
    how='left'
)

dataset['type_inc_7d_sum'] = dataset['type_inc_7d_sum'].fillna(0)
dataset['type_inc_30d_sum'] = dataset['type_inc_30d_sum'].fillna(0)

# Размер инфраструктуры / логарифм хостов

In [249]:
dataset['hosts_log'] = np.log1p(dataset['Количество хостов'])

dataset['Размер инфраструктуры'] = pd.cut(
    dataset['Количество хостов'],
    bins=[-np.inf, 50, 200, 1000, np.inf],
    labels=['small', 'medium', 'large', 'xlarge']
).astype(str)

In [250]:
dataset['entity_day_number'] = dataset.groupby(
    ['infrastructure_cluster', 'threat_cluster']
).cumcount()

dataset_model = dataset[dataset['entity_day_number'] >= 7].copy()

print(dataset_model.shape)

(26112, 36)


In [251]:
dataset_model.columns

Index(['infrastructure_cluster', 'threat_cluster', 'Тип предприятия',
       'Регион размещения предприятия', 'Количество хостов', 'date',
       'incidents_count_day', 'success_count_day', 'had_incident_today',
       'target_next_24h', 'target_next_7d', 'day_of_week', 'day_of_month',
       'month', 'quarter', 'year', 'is_weekend', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos', 'lag_inc_1d', 'lag_inc_2d', 'lag_inc_3d',
       'lag_success_1d', 'lag_success_2d', 'days_since_last_incident',
       'had_success_today', 'days_since_last_success', 'region_inc_7d_sum',
       'region_inc_30d_sum', 'type_inc_7d_sum', 'type_inc_30d_sum',
       'hosts_log', 'Размер инфраструктуры', 'entity_day_number'],
      dtype='object')

# Финальный список признаков

In [252]:
feature_cols = [
    'Тип предприятия',
    'Регион размещения предприятия',
    'Количество хостов',
    'hosts_log',
    'Размер инфраструктуры',

    'day_of_week',
    'day_of_month',
    'month',
    'quarter',
    'year',
    'is_weekend',
    'dow_sin',
    'dow_cos',
    'month_sin',
    'month_cos',

    'lag_inc_1d',
    'lag_inc_2d',
    'lag_inc_3d',
    'lag_success_1d',
    'lag_success_2d',

    #'inc_3d_sum',
    #'inc_7d_sum',
    #'inc_30d_sum',
    #'succ_7d_sum',
    #'succ_30d_sum',
    #'had_incident_prev_1d',
    #'had_incident_prev_3d',
    #'had_incident_prev_7d',

    'days_since_last_incident',
    'days_since_last_success',

    'region_inc_7d_sum',
    'region_inc_30d_sum',
    'type_inc_7d_sum',
    'type_inc_30d_sum'
]

cat_features = [
    'Тип предприятия',
    'Регион размещения предприятия',
    'Размер инфраструктуры'
]

target_col = 'target_next_24h'

model_df = dataset_model[['date'] + feature_cols + [target_col]].copy()

print(model_df.head())
print(model_df[target_col].value_counts(normalize=True))

         date Тип предприятия Регион размещения предприятия  \
7  2023-01-08       маркетинг              Амурская область   
8  2023-01-09       маркетинг              Амурская область   
9  2023-01-10       маркетинг              Амурская область   
10 2023-01-11       маркетинг              Амурская область   
11 2023-01-12       маркетинг              Амурская область   

    Количество хостов  hosts_log Размер инфраструктуры  day_of_week  \
7                1684   7.429521                xlarge            6   
8                1684   7.429521                xlarge            0   
9                1684   7.429521                xlarge            1   
10               1684   7.429521                xlarge            2   
11               1684   7.429521                xlarge            3   

    day_of_month  month  quarter  ...  lag_inc_3d  lag_success_1d  \
7              8      1        1  ...         0.0             0.0   
8              9      1        1  ...         0.0       

# Временной split

In [263]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26112 entries, 0 to 26111
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   date                           26112 non-null  datetime64[ns]
 1   Тип предприятия                26112 non-null  object        
 2   Регион размещения предприятия  26112 non-null  object        
 3   Количество хостов              26112 non-null  int64         
 4   hosts_log                      26112 non-null  float64       
 5   Размер инфраструктуры          26112 non-null  object        
 6   day_of_week                    26112 non-null  int32         
 7   day_of_month                   26112 non-null  int32         
 8   month                          26112 non-null  int32         
 9   quarter                        26112 non-null  int32         
 10  year                           26112 non-null  int32         
 11  is_weekend     

In [254]:
model_df = model_df.sort_values('date').reset_index(drop=True)

unique_dates = model_df['date'].sort_values().unique()

n_dates = len(unique_dates)
train_end = int(n_dates * 0.7)
val_end = int(n_dates * 0.85)

train_dates = unique_dates[:train_end]
val_dates = unique_dates[train_end:val_end]
test_dates = unique_dates[val_end:]

train_df = model_df[model_df['date'].isin(train_dates)].copy()
val_df = model_df[model_df['date'].isin(val_dates)].copy()
test_df = model_df[model_df['date'].isin(test_dates)].copy()

#########
train_df = add_rolling(train_df)
val_df = add_rolling(train_df)
test_df = add_rolling(test_df)

roll =  [
    'inc_3d_sum',
    'inc_7d_sum',
    'inc_30d_sum',
    'succ_7d_sum',
    'succ_30d_sum',
    'had_incident_prev_1d',
    'had_incident_prev_3d',
    'had_incident_prev_7d'
    ]

feature_cols = feature_cols + roll

###########
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.mean(), y_val.mean(), y_test.mean())

(18264, 34) (18264, 34) (3936, 34)
0.07287560227770477 0.07287560227770477 0.07139227642276423


In [255]:
X_test.columns

Index(['Тип предприятия', 'Регион размещения предприятия', 'Количество хостов',
       'hosts_log', 'Размер инфраструктуры', 'day_of_week', 'day_of_month',
       'month', 'quarter', 'year', 'is_weekend', 'dow_sin', 'dow_cos',
       'month_sin', 'month_cos', 'lag_inc_1d', 'lag_inc_2d', 'lag_inc_3d',
       'lag_success_1d', 'lag_success_2d', 'days_since_last_incident',
       'days_since_last_success', 'region_inc_7d_sum', 'region_inc_30d_sum',
       'type_inc_7d_sum', 'type_inc_30d_sum', 'inc_3d_sum', 'inc_7d_sum',
       'inc_30d_sum', 'succ_7d_sum', 'succ_30d_sum', 'had_incident_prev_1d',
       'had_incident_prev_3d', 'had_incident_prev_7d'],
      dtype='object')

In [256]:
model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26112 entries, 0 to 26111
Data columns (total 28 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   date                           26112 non-null  datetime64[ns]
 1   Тип предприятия                26112 non-null  object        
 2   Регион размещения предприятия  26112 non-null  object        
 3   Количество хостов              26112 non-null  int64         
 4   hosts_log                      26112 non-null  float64       
 5   Размер инфраструктуры          26112 non-null  object        
 6   day_of_week                    26112 non-null  int32         
 7   day_of_month                   26112 non-null  int32         
 8   month                          26112 non-null  int32         
 9   quarter                        26112 non-null  int32         
 10  year                           26112 non-null  int32         
 11  is_weekend     

# Обучение CatBoost

In [267]:
import catboost as cb
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

def train_binary_models_by_segments(
    data,
    feature_cols,
    target_col,
    min_rows=120,
    min_positive=10
):
    models = {}
    results = []

    for infra_value in sorted(data['infrastructure_cluster'].astype(str).unique()):
        for threat_value in sorted(data['threat_cluster'].dropna().astype(int).unique()):
            part = data[
                (data['infrastructure_cluster'].astype(str) == str(infra_value)) &
                (data['threat_cluster'].astype(int) == int(threat_value))
            ].copy()

            n_rows = len(part)
            pos_rate = part[target_col].mean() if n_rows > 0 else np.nan
            n_pos = part[target_col].sum() if n_rows > 0 else 0

            print(f"\n{'='*90}")
            print(f"infrastructure_cluster={infra_value}, threat_cluster={threat_value}, target={target_col}")
            print(f"rows={n_rows}, positives={n_pos}, pos_rate={pos_rate:.4f}" if n_rows > 0 else "empty")

            if n_rows < min_rows:
                print("Пропуск: слишком мало строк.")
                continue

            if n_pos < min_positive or n_pos == n_rows:
                print("Пропуск: target слишком несбалансирован / один класс.")
                continue

            part = part.sort_values('date').reset_index(drop=True)

            unique_dates = np.array(sorted(part['date'].unique()))
            n_dates = len(unique_dates)

            if n_dates < 30:
                print("Пропуск: слишком мало уникальных дат.")
                continue

            train_end = int(n_dates * 0.7)
            val_end = int(n_dates * 0.85)

            train_dates = unique_dates[:train_end]
            val_dates = unique_dates[train_end:val_end]
            test_dates = unique_dates[val_end:]

            train_df = part[part['date'].isin(train_dates)].copy()
            val_df   = part[part['date'].isin(val_dates)].copy()
            test_df  = part[part['date'].isin(test_dates)].copy()

            #########
            train_df = add_rolling(train_df)
            val_df = add_rolling(train_df)
            test_df = add_rolling(test_df)

            ###########
            X_train = train_df[feature_cols].copy()
            y_train = train_df[target_col].copy()

            X_val = val_df[feature_cols].copy()
            y_val = val_df[target_col].copy()

            X_test = test_df[feature_cols].copy()
            y_test = test_df[target_col].copy()

            local_cat_features = [c for c in cat_features if c in feature_cols]
            for col in local_cat_features:
                X_train[col] = X_train[col].astype(str)
                X_val[col] = X_val[col].astype(str)
                X_test[col] = X_test[col].astype(str)

            train_pool = cb.Pool(X_train, y_train, cat_features=local_cat_features)
            val_pool = cb.Pool(X_val, y_val, cat_features=local_cat_features)
            test_pool = cb.Pool(X_test, y_test, cat_features=local_cat_features)

            model = cb.CatBoostClassifier(
                loss_function='Logloss',
                eval_metric='PRAUC',
                auto_class_weights='Balanced',
                iterations=1000,
                learning_rate=0.03,
                depth=6,
                l2_leaf_reg=5,
                random_seed=42,
                od_type='Iter',
                od_wait=100,
                verbose=100
            )

            model.fit(
                train_pool,
                eval_set=val_pool,
                use_best_model=True
            )

            val_pred_proba = model.predict_proba(val_pool)[:, 1]
            test_pred_proba = model.predict_proba(test_pool)[:, 1]

            val_pred = (val_pred_proba >= 0.5).astype(int)
            test_pred = (test_pred_proba >= 0.5).astype(int)

            result_row = {
                'infrastructure_cluster': infra_value,
                'threat_cluster': threat_value,
                'target_col': target_col,
                'n_rows': n_rows,
                'train_rows': len(train_df),
                'val_rows': len(val_df),
                'test_rows': len(test_df),
                'train_pos_rate': y_train.mean(),
                'val_pos_rate': y_val.mean(),
                'test_pos_rate': y_test.mean(),
                'val_roc_auc': roc_auc_score(y_val, val_pred_proba),
                'val_pr_auc': average_precision_score(y_val, val_pred_proba),
                'test_roc_auc': roc_auc_score(y_test, test_pred_proba),
                'test_pr_auc': average_precision_score(y_test, test_pred_proba),
                'test_f1': f1_score(y_test, test_pred),
                'best_iteration': model.get_best_iteration(),
                'test_pred_proba' : test_pred_proba
            }

            results.append(result_row)
            models[(str(infra_value), int(threat_value), target_col)] = model

            print("test_pred_proba:", result_row['test_pred_proba'])
            print("VAL ROC-AUC:", result_row['val_roc_auc'])
            print("VAL PR-AUC:", result_row['val_pr_auc'])
            print("TEST ROC-AUC:", result_row['test_roc_auc'])
            print("TEST PR-AUC:", result_row['test_pr_auc'])
            print("TEST F1:", result_row['test_f1'])

    results_df = pd.DataFrame(results)
    return models, results_df

In [268]:
models_24h, results_24h = train_binary_models_by_segments(
    data=dataset_model,
    feature_cols=feature_cols,
    target_col='target_next_24h',
    min_rows=120,
    min_positive=10
)

display(results_24h.sort_values(['test_pr_auc', 'test_roc_auc'], ascending=False))


infrastructure_cluster=1, threat_cluster=1, target=target_next_24h
rows=1088, positives=74, pos_rate=0.0680
0:	learn: 0.6570252	test: 0.6570252	best: 0.6570252 (0)	total: 7.47ms	remaining: 7.46s
100:	learn: 0.9957726	test: 0.9957726	best: 0.9959468 (99)	total: 804ms	remaining: 7.16s
200:	learn: 0.9997993	test: 0.9997993	best: 0.9997993 (200)	total: 1.62s	remaining: 6.45s
300:	learn: 1.0000000	test: 1.0000000	best: 1.0000000 (298)	total: 1.86s	remaining: 4.32s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1
bestIteration = 298

Shrink model to first 299 iterations.
test_pred_proba: [0.030933   0.45112281 0.08683878 0.11765563 0.40262952 0.05474965
 0.09405375 0.04018548 0.17358374 0.21972548 0.06028857 0.19654095
 0.06824499 0.10609014 0.04394525 0.11666062 0.03514213 0.04253571
 0.15664359 0.40754641 0.07591727 0.05291953 0.29679349 0.11497728
 0.12532599 0.39159137 0.42643146 0.03898022 0.10248544 0.47815934
 0.16481159 0.12837545 0.37836646 0.28621861 0.03258232

,infrastructure_cluster,threat_cluster,target_col,n_rows,train_rows,val_rows,test_rows,train_pos_rate,val_pos_rate,test_pos_rate,val_roc_auc,val_pr_auc,test_roc_auc,test_pr_auc,test_f1,best_iteration,test_pred_proba
14,3,3,target_next_24h,1088,761,761,164,0.070959,0.070959,0.079268,1.0,1.0,0.496179,0.158067,0.100000,319,"[0.06783160751536764, 0.19986584939624003, 0.3..."
8,2,3,target_next_24h,1088,761,761,164,0.085414,0.085414,0.103659,1.0,1.0,0.605442,0.153811,0.117647,324,"[0.13479432215613732, 0.11104250595943303, 0.1..."
10,2,5,target_next_24h,1088,761,761,164,0.082786,0.082786,0.073171,1.0,1.0,0.635965,0.149736,0.166667,366,"[0.4884884648862821, 0.5415015994127986, 0.080..."
6,2,1,target_next_24h,1088,761,761,164,0.089356,0.089356,0.085366,1.0,1.0,0.584286,0.138664,0.000000,458,"[0.28471894347100757, 0.01722963387936187, 0.0..."
23,4,6,target_next_24h,1088,761,761,164,0.053876,0.053876,0.054878,1.0,1.0,0.696057,0.119112,0.111111,265,"[0.018662534948078357, 0.16986838145079705, 0...."
18,4,1,target_next_24h,1088,761,761,164,0.074901,0.074901,0.109756,1.0,1.0,0.471842,0.112559,0.108108,352,"[0.21283311168247127, 0.3461403387643979, 0.24..."
11,2,6,target_next_24h,1088,761,761,164,0.059133,0.059133,0.067073,1.0,1.0,0.651812,0.107981,0.000000,240,"[0.10543858075009799, 0.20299388891183812, 0.2..."
12,3,1,target_next_24h,1088,761,761,164,0.073587,0.073587,0.067073,1.0,1.0,0.516340,0.107012,0.148148,270,"[0.2696865256018688, 0.20858616238859828, 0.26..."
16,3,5,target_next_24h,1088,761,761,164,0.069645,0.069645,0.091463,1.0,1.0,0.557047,0.102505,0.000000,238,"[0.4915721892086452, 0.4066507919233573, 0.443..."
20,4,3,target_next_24h,1088,761,761,164,0.064389,0.064389,0.091463,1.0,1.0,0.511409,0.095559,0.000000,313,"[0.32033772759572365, 0.21019763272426725, 0.3..."


In [264]:
dataset_model.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26112 entries, 7 to 26279
Data columns (total 36 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   infrastructure_cluster         26112 non-null  object        
 1   threat_cluster                 26112 non-null  int64         
 2   Тип предприятия                26112 non-null  object        
 3   Регион размещения предприятия  26112 non-null  object        
 4   Количество хостов              26112 non-null  int64         
 5   date                           26112 non-null  datetime64[ns]
 6   incidents_count_day            26112 non-null  int64         
 7   success_count_day              26112 non-null  int64         
 8   had_incident_today             26112 non-null  int64         
 9   target_next_24h                26112 non-null  int64         
 10  target_next_7d                 26112 non-null  int64         
 11  day_of_week         

In [269]:
models_7d, results_7d = train_binary_models_by_segments(
    data=dataset_model,
    feature_cols=feature_cols,
    target_col='target_next_7d',
    min_rows=120,
    min_positive=10
)

display(results_7d.sort_values(['test_pr_auc', 'test_roc_auc'], ascending=False))


infrastructure_cluster=1, threat_cluster=1, target=target_next_7d
rows=1088, positives=417, pos_rate=0.3833
0:	learn: 0.6805933	test: 0.6805933	best: 0.6805933 (0)	total: 1.71ms	remaining: 1.7s
100:	learn: 0.9837492	test: 0.9837492	best: 0.9838127 (99)	total: 179ms	remaining: 1.59s
200:	learn: 0.9929240	test: 0.9929240	best: 0.9929240 (200)	total: 325ms	remaining: 1.29s
300:	learn: 0.9987137	test: 0.9987137	best: 0.9987137 (300)	total: 497ms	remaining: 1.15s
400:	learn: 0.9999705	test: 0.9999705	best: 0.9999705 (392)	total: 642ms	remaining: 959ms
500:	learn: 1.0000000	test: 1.0000000	best: 1.0000000 (411)	total: 792ms	remaining: 789ms
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1
bestIteration = 411

Shrink model to first 412 iterations.
test_pred_proba: [0.39589965 0.33134879 0.37324731 0.53536432 0.7161618  0.52109941
 0.50852314 0.30235408 0.24616793 0.37445604 0.41000365 0.46143325
 0.27968275 0.29775432 0.66146722 0.5333938  0.4430275  0.45670176
 0.6525381

,infrastructure_cluster,threat_cluster,target_col,n_rows,train_rows,val_rows,test_rows,train_pos_rate,val_pos_rate,test_pos_rate,val_roc_auc,val_pr_auc,test_roc_auc,test_pr_auc,test_f1,best_iteration,test_pred_proba
18,4,1,target_next_7d,1088,761,761,164,0.440210,0.440210,0.609756,1.0,1.0,0.543438,0.665288,0.660465,548,"[0.3185787975331748, 0.37899785704550026, 0.52..."
15,3,4,target_next_7d,1088,761,761,164,0.509855,0.509855,0.231707,1.0,1.0,0.798454,0.571578,0.417722,638,"[0.18785622546019498, 0.07722942090404972, 0.0..."
13,3,2,target_next_7d,1088,761,761,164,0.469120,0.469120,0.475610,1.0,1.0,0.611956,0.569984,0.426230,463,"[0.8201978886955857, 0.8263918930801022, 0.807..."
8,2,3,target_next_7d,1088,761,761,164,0.438896,0.438896,0.518293,1.0,1.0,0.556813,0.541004,0.527607,486,"[0.10767572222927697, 0.0879257744247389, 0.18..."
6,2,1,target_next_7d,1088,761,761,164,0.504599,0.504599,0.536585,1.0,1.0,0.481609,0.535811,0.308943,477,"[0.1290089989823395, 0.07527401553689081, 0.10..."
4,1,5,target_next_7d,1088,761,761,164,0.429698,0.429698,0.347561,1.0,1.0,0.625184,0.534251,0.500000,669,"[0.04789098784721435, 0.09457883470638392, 0.0..."
21,4,4,target_next_7d,1088,761,761,164,0.390276,0.390276,0.365854,1.0,1.0,0.595353,0.531800,0.440367,511,"[0.07593327827462992, 0.060066726517894437, 0...."
23,4,6,target_next_7d,1088,761,761,164,0.318003,0.318003,0.298780,1.0,1.0,0.628926,0.502883,0.428571,511,"[0.06862143957801896, 0.04979796323694307, 0.0..."
7,2,2,target_next_7d,1088,761,761,164,0.436268,0.436268,0.396341,1.0,1.0,0.505206,0.457860,0.437086,704,"[0.11613932822420864, 0.06901301445527196, 0.1..."
20,4,3,target_next_7d,1088,761,761,164,0.357424,0.357424,0.518293,1.0,1.0,0.448548,0.456458,0.000000,473,"[0.2770406439746265, 0.25482475732826365, 0.21..."


# Сохранение

In [270]:
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

MVP_DIR = PROJECT_ROOT / "mvp"
MODELS_24H_DIR = MVP_DIR / "models" / "24h"
MODELS_7D_DIR = MVP_DIR / "models" / "7d"
DATA_DIR = MVP_DIR / "data"

for path in [MODELS_24H_DIR, MODELS_7D_DIR, DATA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_24H = "target_next_24h"
TARGET_7D = "target_next_7d"

clusterinfo = {
    "infrastructure_clusters": {
        "1": "Digital-Native (Высокотехнологичные)",
        "2": "Industrial IoT (Промышленные)",
        "3": "Data-Sensitive (Чувствительные данные)",
        "4": "Service-Oriented (Сервисные)"
    },
    "threat_clusters": {
        "1": "Вредоносное ПО / Malware",
        "2": "Атаки типа DDoS",
        "3": "Brute Force / Подбор паролей",
        "4": "Социальная инженерия / Фишинг",
        "5": "Эксплуатация уязвимостей",
        "6": "Инсайдерские угрозы"
    }
}

threatdescriptions = {
    "1": {
        "name": "Вредоносное ПО (Malware)",
        "description": "Распространение вредоносного ПО, включая вирусы, трояны и ransomware.",
        "recommendation": "Обновить антивирусные базы, провести сканирование узлов, изолировать зараженные системы.",
        "mitigation_steps": [
            "Установить обновления безопасности",
            "Включить поведенческий анализ",
            "Ограничить выполнение макросов"
        ]
    },
    "2": {
        "name": "Атаки типа DDoS",
        "description": "Распределенные атаки на отказ в обслуживании, перегружающие сетевую инфраструктуру.",
        "recommendation": "Усилить фильтрацию трафика, включить DDoS-защиту, настроить rate limiting.",
        "mitigation_steps": [
            "Включить DDoS-защиту провайдера",
            "Настроить черные списки IP",
            "Увеличить пропускную способность"
        ]
    },
    "3": {
        "name": "Brute Force / Подбор паролей",
        "description": "Попытки подбора учетных данных через словарные атаки и перебор.",
        "recommendation": "Включить MFA, ограничить число попыток входа, использовать CAPTCHA.",
        "mitigation_steps": [
            "Внедрить MFA",
            "Настроить блокировку после N попыток",
            "Использовать сложные пароли"
        ]
    },
    "4": {
        "name": "Социальная инженерия / Фишинг",
        "description": "Фишинговые атаки на сотрудников с целью кражи учетных данных и доступа.",
        "recommendation": "Провести обучение сотрудников и внедрить антифишинговую защиту.",
        "mitigation_steps": [
            "Обучение сотрудников",
            "Фильтрация входящей почты",
            "Внедрение DMARC"
        ]
    },
    "5": {
        "name": "Эксплуатация уязвимостей",
        "description": "Попытки использования известных уязвимостей в ПО и инфраструктуре.",
        "recommendation": "Установить критические патчи и провести сканирование уязвимостей.",
        "mitigation_steps": [
            "Регулярное обновление ПО",
            "Внедрение системы управления уязвимостями",
            "Сегментация сети"
        ]
    },
    "6": {
        "name": "Инсайдерские угрозы",
        "description": "Несанкционированные действия сотрудников или подрядчиков.",
        "recommendation": "Пересмотреть права доступа и усилить контроль привилегированных пользователей.",
        "mitigation_steps": [
            "Принцип наименьших привилегий",
            "Мониторинг действий пользователей",
            "Использование DLP-систем"
        ]
    }
}


def save_segment_models(models_dict, target_name, save_dir, relative_prefix):
    saved_files = []

    for (infra, threat, target), model in models_dict.items():
        if target != target_name:
            continue

        infra = str(infra)
        threat = str(threat)

        filename = f"model_infra{infra}_threat{threat}.cbm"
        model_path = save_dir / filename
        model.save_model(str(model_path))

        saved_files.append({
            "infrastructure_cluster": infra,
            "threat_cluster": threat,
            "target": target,
            "model_file": filename,
            "model_path": f"{relative_prefix}/{filename}"
        })

    return saved_files


print("Сохраняем модели для прогноза на 24 часа...")
saved24h = save_segment_models(
    models_dict=models_24h,
    target_name=TARGET_24H,
    save_dir=MODELS_24H_DIR,
    relative_prefix="models/24h"
)

print("Сохраняем модели для прогноза на 7 дней...")
saved7d = save_segment_models(
    models_dict=models_7d,
    target_name=TARGET_7D,
    save_dir=MODELS_7D_DIR,
    relative_prefix="models/7d"
)

results_24h.to_csv(DATA_DIR / "results24h.csv", index=False)
results_7d.to_csv(DATA_DIR / "results7d.csv", index=False)

resolved_featurecols = list(featurecols) if "featurecols" in globals() else []
resolved_catfeatures = list(catfeatures) if "catfeatures" in globals() else []

featureconfig = {
    "featurecols": feature_cols,
    "catfeatures": cat_features,
    "targets": {
        "24h": TARGET_24H,
        "7d": TARGET_7D
    }
}

with open(DATA_DIR / "featureconfig.json", "w", encoding="utf-8") as f:
    json.dump(featureconfig, f, ensure_ascii=False, indent=2)

with open(DATA_DIR / "clusterinfo.json", "w", encoding="utf-8") as f:
    json.dump(clusterinfo, f, ensure_ascii=False, indent=2)

with open(DATA_DIR / "threatdescriptions.json", "w", encoding="utf-8") as f:
    json.dump(threatdescriptions, f, ensure_ascii=False, indent=2)

model_registry = {
    "24h": saved24h,
    "7d": saved7d
}

with open(DATA_DIR / "model_registry.json", "w", encoding="utf-8") as f:
    json.dump(model_registry, f, ensure_ascii=False, indent=2)

if "datasetmodel" in globals():
    sample_columns = resolved_featurecols + ["date", "infrastructure_cluster", "threat_cluster"]
    sample_columns = [col for col in sample_columns if col in datasetmodel.columns]

    if sample_columns:
        sample_features = datasetmodel[sample_columns].head(100).copy()
        sample_features.to_csv(DATA_DIR / "sample_features.csv", index=False)

    required_data_columns = [
        "date",
        "infrastructure_cluster",
        "threat_cluster",
        "incidents_count_day",
        "success_count_day"
    ]
    available_required_columns = [col for col in required_data_columns if col in datasetmodel.columns]

    if available_required_columns:
        datasetmodel[available_required_columns].to_csv(
            DATA_DIR / "incidents_data.csv",
            index=False
        )
else:
    print("datasetmodel не найден: sample_features.csv и incidents_data.csv не сохранены")

dataset_model.to_csv(DATA_DIR / "incidents_data.csv", index=False)

print(f"Готово. Сохранено моделей 24h: {len(saved24h)}, моделей 7d: {len(saved7d)}")
print(f"Артефакты сохранены в: {MVP_DIR}")

Сохраняем модели для прогноза на 24 часа...
Сохраняем модели для прогноза на 7 дней...
datasetmodel не найден: sample_features.csv и incidents_data.csv не сохранены
Готово. Сохранено моделей 24h: 24, моделей 7d: 24
Артефакты сохранены в: /content/mvp


In [271]:
!zip -r mvp_artifacts.zip mvp

  adding: mvp/ (stored 0%)
  adding: mvp/models/ (stored 0%)
  adding: mvp/models/24h/ (stored 0%)
  adding: mvp/models/24h/model_infra2_threat6.cbm (deflated 67%)
  adding: mvp/models/24h/model_infra1_threat2.cbm (deflated 68%)
  adding: mvp/models/24h/model_infra1_threat6.cbm (deflated 68%)
  adding: mvp/models/24h/model_infra1_threat3.cbm (deflated 66%)
  adding: mvp/models/24h/model_infra4_threat6.cbm (deflated 68%)
  adding: mvp/models/24h/model_infra3_threat5.cbm (deflated 67%)
  adding: mvp/models/24h/model_infra2_threat1.cbm (deflated 66%)
  adding: mvp/models/24h/model_infra1_threat4.cbm (deflated 65%)
  adding: mvp/models/24h/model_infra3_threat4.cbm (deflated 65%)
  adding: mvp/models/24h/model_infra4_threat1.cbm (deflated 69%)
  adding: mvp/models/24h/model_infra2_threat5.cbm (deflated 68%)
  adding: mvp/models/24h/model_infra3_threat1.cbm (deflated 65%)
  adding: mvp/models/24h/model_infra2_threat4.cbm (deflated 68%)
  adding: mvp/models/24h/model_infra1_threat1.cbm (defla

In [272]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [273]:
!cp -r /content/mvp /content/drive/MyDrive/mvp_rsm